In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [11]:
!pip -q install pandas numpy scikit-learn sentence-transformers rank-bm25 openpyxl underthesea transformers accelerate bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.0/7.0 MB 79.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 15.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 91.1 MB/s eta 0:00:00


## Evaluation Retrieval

In [ ]:
# import os
# os.kill(os.getpid(), 9)

In [ ]:
import json
import re
from pathlib import Path
from typing import Any, Dict

import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import minmax_scale

from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer
from underthesea import word_tokenize

In [ ]:
# ===== CONFIG =====
KG_DIR = Path("/content/drive/MyDrive/CS2231/kg")
KG_NODES_FILE = KG_DIR / "kg_nodes.jsonl"
TEST_CSV = KG_DIR / "test.csv"
OUTPUT_XLSX = KG_DIR / "retrieval_method_comparison_full.xlsx"

EMB_CACHE_DIR = KG_DIR / "embedding_cache"
EMB_CACHE_DIR.mkdir(parents=True, exist_ok=True)

E5_MODEL_NAME = "intfloat/multilingual-e5-large"
MiniLM_L12_MODEL_NAME = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
BGE_M3_MODEL_NAME = "BAAI/bge-m3"
TOP_K = 10

# Hybrid weights
TFIDF_EMB_ALPHA = 0.5   # final = alpha * tfidf + (1-alpha) * embedding
BM25_EMB_ALPHA = 0.5    # final = alpha * bm25 + (1-alpha) * embedding

In [ ]:
def get_query_prefix(model_name: str) -> str:
    if "e5" in model_name.lower():
        return "query: "
    return ""

def get_passage_prefix(model_name: str) -> str:
    if "e5" in model_name.lower():
        return "passage: "
    return ""

def build_or_load_embeddings(model_name: str, corpus_texts, query_texts, cache_prefix: str):
    model = SentenceTransformer(model_name)

    safe_model_name = (
        model_name.replace("/", "_")
        .replace("-", "_")
        .replace(".", "_")
    )

    doc_emb_path = EMB_CACHE_DIR / f"{cache_prefix}_{safe_model_name}_doc.npy"
    query_emb_path = EMB_CACHE_DIR / f"{cache_prefix}_{safe_model_name}_query.npy"

    passage_prefix = get_passage_prefix(model_name)
    query_prefix = get_query_prefix(model_name)

    passages = [f"{passage_prefix}{text}" for text in corpus_texts]
    queries = [f"{query_prefix}{q}" for q in query_texts]

    if doc_emb_path.exists():
        print(f"Loading cached doc embeddings for {model_name} ...")
        doc_embeddings = np.load(doc_emb_path)
    else:
        print(f"Encoding doc embeddings for {model_name} ...")
        doc_embeddings = model.encode(
            passages,
            normalize_embeddings=True,
            convert_to_numpy=True,
            batch_size=32,
            show_progress_bar=True
        ).astype("float32")
        np.save(doc_emb_path, doc_embeddings)

    if query_emb_path.exists():
        print(f"Loading cached query embeddings for {model_name} ...")
        query_embeddings = np.load(query_emb_path)
    else:
        print(f"Encoding query embeddings for {model_name} ...")
        query_embeddings = model.encode(
            queries,
            normalize_embeddings=True,
            convert_to_numpy=True,
            batch_size=32,
            show_progress_bar=True
        ).astype("float32")
        np.save(query_emb_path, query_embeddings)

    return doc_embeddings, query_embeddings

In [ ]:
# ===== LOAD TEST =====
df_eval = pd.read_csv(TEST_CSV)
df_eval["violation_id"] = df_eval["violation_id"].astype(str)
df_eval["query"] = df_eval["query"].astype(str)

print("Test size:", len(df_eval))
display(df_eval.head())

y_true = df_eval["violation_id"].tolist()

Test size: 576


,query,behavior,legal,fine_min,fine_max,violation_id
0,Tôi muốn tra lỗi xe thô sơ: không có đèn chiếu...,không có đèn chiếu sáng 40 hoặc tấm phản quang...,corpus://dataset-traffic-law#dieu-15-khoan-1-d...,100000,200000,V001
1,Cho tôi hỏi hành vi Không chấp hành hướng dẫn ...,"Không chấp hành hướng dẫn của lái xe, nhân viê...",corpus://dataset-traffic-law#dieu-33-khoan-1-d...,100000,200000,V002
2,Tra cứu vi phạm giao thông: xe đạp Không đi bê...,"Không đi bên phải theo chiều đi của mình, đi k...",corpus://dataset-traffic-law#dieu-9-khoan-1-di...,100000,200000,V003
3,Trường hợp xe đạp Dừng xe đột ngột thì bị xử p...,Dừng xe đột ngột; chuyển hướng không báo hiệu ...,corpus://dataset-traffic-law#dieu-9-khoan-1-di...,100000,200000,V004
4,Hỏi nhanh lỗi Không chấp hành hiệu lệnh hoặc c...,Không chấp hành hiệu lệnh hoặc chỉ dẫn của biể...,corpus://dataset-traffic-law#dieu-9-khoan-1-di...,100000,200000,V005


In [ ]:
def hit_at_k(ranks, k):
    return sum(1 for r in ranks if r is not None and r <= k) / len(ranks)

def mrr_at_k(ranks, k):
    return sum((1 / r) if r is not None and r <= k else 0 for r in ranks) / len(ranks)

def compute_metrics(y_true, topk_preds, method_name):
    best_preds = [preds[0] if len(preds) > 0 else "NOT_FOUND" for preds in topk_preds]
    ranks = []

    for true_id, preds in zip(y_true, topk_preds):
        if true_id in preds:
            ranks.append(preds.index(true_id) + 1)
        else:
            ranks.append(None)

    accuracy = accuracy_score(y_true, best_preds)
    hit3 = hit_at_k(ranks, 3)
    hit5 = hit_at_k(ranks, 5)
    hit10 = hit_at_k(ranks, 10)
    mrr10 = mrr_at_k(ranks, 10)

    summary_row = {
        "model": method_name,
        "Accuracy": accuracy,
        "Hit@3": hit3,
        "Hit@5": hit5,
        "Hit@10": hit10,
        "MRR@10": mrr10,
    }
    return summary_row, best_preds, ranks

def build_detail_df(df_eval, topk_preds, best_preds, ranks, top_k=10):
    details_df = df_eval.copy()
    details_df["violation_id"] = details_df["violation_id"].astype(str)
    details_df["top_1_pred"] = best_preds
    details_df["rank_of_true_in_top10"] = ranks

    for i in range(top_k):
        details_df[f"top_{i+1}"] = [
            preds[i] if i < len(preds) else "" for preds in topk_preds
        ]

    details_df["is_correct_top1"] = details_df["violation_id"].astype(str) == details_df["top_1_pred"].astype(str)
    details_df["hit@3"] = details_df["rank_of_true_in_top10"].apply(lambda x: x is not None and x <= 3)
    details_df["hit@5"] = details_df["rank_of_true_in_top10"].apply(lambda x: x is not None and x <= 5)
    details_df["hit@10"] = details_df["rank_of_true_in_top10"].apply(lambda x: x is not None and x <= 10)
    details_df["rr@10"] = details_df["rank_of_true_in_top10"].apply(lambda x: (1 / x) if x is not None and x <= 10 else 0)
    return details_df

def print_summary(summary_row):
    print("=" * 50)
    print(summary_row["model"])
    print("=" * 50)
    print(f'Accuracy : {summary_row["Accuracy"]:.4f}')
    print(f'Hit@3    : {summary_row["Hit@3"]:.4f}')
    print(f'Hit@5    : {summary_row["Hit@5"]:.4f}')
    print(f'Hit@10   : {summary_row["Hit@10"]:.4f}')
    print(f'MRR@10   : {summary_row["MRR@10"]:.4f}')

def safe_str(x):
    if x is None:
        return ""
    return str(x).strip()

#### TF-IDF FUNCTION

In [ ]:
# ===== TF-IDF FUNCTIONS =====
def norm_tfidf(text: Any) -> str:
    if text is None:
        return ""
    text = str(text).lower().strip()
    text = text.replace("đ", "d")
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def tokenize_tfidf(text: str):
    return norm_tfidf(text).split()

def make_violation_text_tfidf(node: Dict[str, Any]) -> str:
    parts = [
        safe_str(node.get("description_natural")),
        safe_str(node.get("normalized_violation")),
        safe_str(node.get("vehicle_type")),
        safe_str(node.get("context_condition")),
        safe_str(node.get("legal_basis")),
        safe_str(node.get("additional_sanctions")),
    ]
    return " | ".join([p for p in parts if p])

In [ ]:
# ===== BUILD TF-IDF CORPUS =====
violation_rows_tfidf = []

with open(KG_NODES_FILE, "r", encoding="utf-8") as f:
    for line in f:
        node = json.loads(line)
        if node.get("type") != "Violation":
            continue

        violation_id = safe_str(node.get("violation_id"))
        if not violation_id:
            continue

        text = make_violation_text_tfidf(node)

        violation_rows_tfidf.append({
            "node_id": node.get("id"),
            "violation_id": violation_id,
            "text": text,
            "norm_text": norm_tfidf(text),
            "raw_node": node,
        })

df_corpus_tfidf = (
    pd.DataFrame(violation_rows_tfidf)
    .drop_duplicates(subset=["violation_id"])
    .reset_index(drop=True)
)

print("TF-IDF corpus size:", len(df_corpus_tfidf))
display(df_corpus_tfidf.head())

corpus_ids_tfidf = df_corpus_tfidf["violation_id"].tolist()
corpus_texts_tfidf = df_corpus_tfidf["text"].tolist()
corpus_norm_texts_tfidf = df_corpus_tfidf["norm_text"].tolist()

TF-IDF corpus size: 613


,node_id,violation_id,text,norm_text,raw_node
0,VIOLATION_V001,V001,không có đèn chiếu sáng 40 hoặc tấm phản quang...,kh ng c d n chi u s ng 40 ho c t m ph n quang ...,"{'id': 'VIOLATION_V001', 'type': 'Violation', ..."
1,VIOLATION_V002,V002,"Không chấp hành hướng dẫn của lái xe, nhân viê...",kh ng ch p h nh h ng d n c a l i xe nh n vi n ...,"{'id': 'VIOLATION_V002', 'type': 'Violation', ..."
2,VIOLATION_V003,V003,"Không đi bên phải theo chiều đi của mình, đi k...",kh ng di b n ph i theo chi u di c a m nh di kh...,"{'id': 'VIOLATION_V003', 'type': 'Violation', ..."
3,VIOLATION_V004,V004,Dừng xe đột ngột; chuyển hướng không báo hiệu ...,d ng xe d t ng t chuy n h ng kh ng b o hi u tr...,"{'id': 'VIOLATION_V004', 'type': 'Violation', ..."
4,VIOLATION_V005,V005,Không chấp hành hiệu lệnh hoặc chỉ dẫn của biể...,kh ng ch p h nh hi u l nh ho c ch d n c a bi n...,"{'id': 'VIOLATION_V005', 'type': 'Violation', ..."


#### BM25 FUNCTION

In [ ]:
# ===== BM25 FUNCTIONS =====
def norm_bm25(text):
    if text is None:
        return ""
    text = str(text).lower().strip()
    text = re.sub(r"[^\w\s]", " ", text, flags=re.UNICODE)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def tokenize_bm25(text: str):
    text = norm_bm25(text)
    segmented = word_tokenize(text, format="text")
    return segmented.split()

def make_violation_text_bm25(node):
    parts = [
        safe_str(node.get("description_natural")),
        safe_str(node.get("normalized_violation")),
        safe_str(node.get("vehicle_type")),
        safe_str(node.get("context_condition")),
    ]
    return " ".join([p for p in parts if p])

In [ ]:
# ===== BUILD BM25 / EMBEDDING CORPUS =====
violation_rows_bm25 = []

with open(KG_NODES_FILE, "r", encoding="utf-8") as f:
    for line in f:
        node = json.loads(line)
        if node.get("type") != "Violation":
            continue

        violation_id = safe_str(node.get("violation_id"))
        if not violation_id:
            continue

        text = make_violation_text_bm25(node)

        violation_rows_bm25.append({
            "node_id": node.get("id"),
            "violation_id": violation_id,
            "text": text,
            "raw_node": node,
        })

df_corpus_bm25 = (
    pd.DataFrame(violation_rows_bm25)
    .drop_duplicates(subset=["violation_id"])
    .reset_index(drop=True)
)

print("BM25/Embedding corpus size:", len(df_corpus_bm25))
display(df_corpus_bm25.head())

corpus_ids_bm25 = df_corpus_bm25["violation_id"].tolist()
corpus_texts_bm25 = df_corpus_bm25["text"].tolist()

BM25/Embedding corpus size: 613


,node_id,violation_id,text,raw_node
0,VIOLATION_V001,V001,không có đèn chiếu sáng 40 hoặc tấm phản quang...,"{'id': 'VIOLATION_V001', 'type': 'Violation', ..."
1,VIOLATION_V002,V002,"Không chấp hành hướng dẫn của lái xe, nhân viê...","{'id': 'VIOLATION_V002', 'type': 'Violation', ..."
2,VIOLATION_V003,V003,"Không đi bên phải theo chiều đi của mình, đi k...","{'id': 'VIOLATION_V003', 'type': 'Violation', ..."
3,VIOLATION_V004,V004,Dừng xe đột ngột; chuyển hướng không báo hiệu ...,"{'id': 'VIOLATION_V004', 'type': 'Violation', ..."
4,VIOLATION_V005,V005,Không chấp hành hiệu lệnh hoặc chỉ dẫn của biể...,"{'id': 'VIOLATION_V005', 'type': 'Violation', ..."


#### TF-IDF

In [ ]:
# ===== TF-IDF =====
tfidf_vectorizer = TfidfVectorizer(
    lowercase=True,
    ngram_range=(1, 2),
    min_df=1,
    token_pattern=r"(?u)\b\w+\b"
)

X_tfidf = tfidf_vectorizer.fit_transform(corpus_norm_texts_tfidf)

tfidf_topk_preds = []
tfidf_all_scores = []

for query in df_eval["query"].tolist():
    q_norm = norm_tfidf(query)
    q_vec = tfidf_vectorizer.transform([q_norm])

    scores = (q_vec @ X_tfidf.T).toarray()[0]
    tfidf_all_scores.append(scores)

    ranked_idx = np.argsort(scores)[::-1][:TOP_K]
    pred_ids = [corpus_ids_tfidf[i] for i in ranked_idx]
    tfidf_topk_preds.append(pred_ids)

tfidf_summary, tfidf_best_preds, tfidf_ranks = compute_metrics(
    y_true, tfidf_topk_preds, "TF-IDF"
)
tfidf_details = build_detail_df(df_eval, tfidf_topk_preds, tfidf_best_preds, tfidf_ranks, TOP_K)

print_summary(tfidf_summary)

TF-IDF
Accuracy : 0.8750
Hit@3    : 0.9792
Hit@5    : 0.9983
Hit@10   : 1.0000
MRR@10   : 0.9286


#### BM25

In [ ]:
# ===== BM25 =====
tokenized_corpus_bm25 = [tokenize_bm25(text) for text in corpus_texts_bm25]
bm25 = BM25Okapi(tokenized_corpus_bm25)

bm25_topk_preds = []
bm25_all_scores = []

for query in df_eval["query"].tolist():
    tokenized_query = tokenize_bm25(query)
    scores = bm25.get_scores(tokenized_query)
    bm25_all_scores.append(scores)

    ranked_idx = np.argsort(scores)[::-1][:TOP_K]
    pred_ids = [corpus_ids_bm25[i] for i in ranked_idx]
    bm25_topk_preds.append(pred_ids)

bm25_summary, bm25_best_preds, bm25_ranks = compute_metrics(
    y_true, bm25_topk_preds, "BM25"
)
bm25_details = build_detail_df(df_eval, bm25_topk_preds, bm25_best_preds, bm25_ranks, TOP_K)

print_summary(bm25_summary)

BM25
Accuracy : 0.8958
Hit@3    : 0.9878
Hit@5    : 0.9983
Hit@10   : 1.0000
MRR@10   : 0.9424


#### Embedding (e5-large)

In [ ]:
# ===== EMBEDDING: E5 =====
doc_embeddings_e5, query_embeddings_e5 = build_or_load_embeddings(
    model_name=E5_MODEL_NAME,
    corpus_texts=corpus_texts_bm25,
    query_texts=df_eval["query"].tolist(),
    cache_prefix="bm25_corpus"
)

embedding_topk_preds = []
embedding_all_scores = []

for q_emb in query_embeddings_e5:
    scores = np.dot(doc_embeddings_e5, q_emb)
    embedding_all_scores.append(scores)

    ranked_idx = np.argsort(scores)[::-1][:TOP_K]
    pred_ids = [corpus_ids_bm25[i] for i in ranked_idx]
    embedding_topk_preds.append(pred_ids)

embedding_summary, embedding_best_preds, embedding_ranks = compute_metrics(
    y_true, embedding_topk_preds, f"Embedding ({E5_MODEL_NAME})"
)
embedding_details = build_detail_df(df_eval, embedding_topk_preds, embedding_best_preds, embedding_ranks, TOP_K)

print_summary(embedding_summary)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: intfloat/multilingual-e5-large
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading cached doc embeddings for intfloat/multilingual-e5-large ...
Loading cached query embeddings for intfloat/multilingual-e5-large ...
Embedding (intfloat/multilingual-e5-large)
Accuracy : 0.8299
Hit@3    : 0.9844
Hit@5    : 0.9931
Hit@10   : 0.9965
MRR@10   : 0.9055


In [ ]:
# ===== EMBEDDING: MiniLM-L12 =====
doc_embeddings_minilm, query_embeddings_minilm = build_or_load_embeddings(
    model_name=MiniLM_L12_MODEL_NAME,
    corpus_texts=corpus_texts_bm25,
    query_texts=df_eval["query"].tolist(),
    cache_prefix="bm25_corpus"
)

minilm_topk_preds = []
minilm_all_scores = []

for q_emb in query_embeddings_minilm:
    scores = np.dot(doc_embeddings_minilm, q_emb)
    minilm_all_scores.append(scores)

    ranked_idx = np.argsort(scores)[::-1][:TOP_K]
    pred_ids = [corpus_ids_bm25[i] for i in ranked_idx]
    minilm_topk_preds.append(pred_ids)

minilm_summary, minilm_best_preds, minilm_ranks = compute_metrics(
    y_true, minilm_topk_preds, f"Embedding ({MiniLM_L12_MODEL_NAME})"
)
minilm_details = build_detail_df(df_eval, minilm_topk_preds, minilm_best_preds, minilm_ranks, TOP_K)

print_summary(minilm_summary)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading cached doc embeddings for sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2 ...
Loading cached query embeddings for sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2 ...
Embedding (sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2)
Accuracy : 0.5538
Hit@3    : 0.7552
Hit@5    : 0.8194
Hit@10   : 0.8681
MRR@10   : 0.6659


In [ ]:
# ===== EMBEDDING: BGE-M3 =====
doc_embeddings_bge_m3, query_embeddings_bge_m3 = build_or_load_embeddings(
    model_name=BGE_M3_MODEL_NAME,
    corpus_texts=corpus_texts_bm25,
    query_texts=df_eval["query"].tolist(),
    cache_prefix="bm25_corpus"
)

bge_m3_topk_preds = []
bge_m3_all_scores = []

for q_emb in query_embeddings_bge_m3:
    scores = np.dot(doc_embeddings_bge_m3, q_emb)
    bge_m3_all_scores.append(scores)

    ranked_idx = np.argsort(scores)[::-1][:TOP_K]
    pred_ids = [corpus_ids_bm25[i] for i in ranked_idx]
    bge_m3_topk_preds.append(pred_ids)

bge_m3_summary, bge_m3_best_preds, bge_m3_ranks = compute_metrics(
    y_true, bge_m3_topk_preds, f"Embedding ({BGE_M3_MODEL_NAME})"
)
bge_m3_details = build_detail_df(df_eval, bge_m3_topk_preds, bge_m3_best_preds, bge_m3_ranks, TOP_K)

print_summary(bge_m3_summary)

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Loading cached doc embeddings for BAAI/bge-m3 ...
Loading cached query embeddings for BAAI/bge-m3 ...
Embedding (BAAI/bge-m3)
Accuracy : 0.8212
Hit@3    : 0.9497
Hit@5    : 0.9722
Hit@10   : 0.9878
MRR@10   : 0.8875


#### TF-IDF + Embedding

In [ ]:
# ===== TF-IDF + EMBEDDING (E5 + BGE-M3) =====
# Hybrid trên corpus TF-IDF để score align đúng index

# --- E5 embeddings on TF-IDF corpus ---
doc_embeddings_tfidf_e5, query_embeddings_tfidf_e5 = build_or_load_embeddings(
    model_name=E5_MODEL_NAME,
    corpus_texts=corpus_texts_tfidf,
    query_texts=df_eval["query"].tolist(),
    cache_prefix="tfidf_corpus"
)

tfidf_e5_topk_preds = []

for i in range(len(df_eval)):
    emb_scores = np.dot(doc_embeddings_tfidf_e5, query_embeddings_tfidf_e5[i])
    tfidf_scores = tfidf_all_scores[i]

    tfidf_norm = minmax_scale(tfidf_scores) if np.max(tfidf_scores) != np.min(tfidf_scores) else np.zeros_like(tfidf_scores)
    emb_norm = minmax_scale(emb_scores) if np.max(emb_scores) != np.min(emb_scores) else np.zeros_like(emb_scores)

    final_scores = TFIDF_EMB_ALPHA * tfidf_norm + (1 - TFIDF_EMB_ALPHA) * emb_norm

    ranked_idx = np.argsort(final_scores)[::-1][:TOP_K]
    pred_ids = [corpus_ids_tfidf[idx] for idx in ranked_idx]
    tfidf_e5_topk_preds.append(pred_ids)

tfidf_e5_summary, tfidf_e5_best_preds, tfidf_e5_ranks = compute_metrics(
    y_true,
    tfidf_e5_topk_preds,
    f"TF-IDF + Embedding ({E5_MODEL_NAME}, alpha={TFIDF_EMB_ALPHA})"
)

tfidf_e5_details = build_detail_df(
    df_eval,
    tfidf_e5_topk_preds,
    tfidf_e5_best_preds,
    tfidf_e5_ranks,
    TOP_K
)

print_summary(tfidf_e5_summary)

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: intfloat/multilingual-e5-large
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading cached doc embeddings for intfloat/multilingual-e5-large ...
Encoding query embeddings for intfloat/multilingual-e5-large ...


Batches:   0%|          | 0/18 [00:00<?, ?it/s]

TF-IDF + Embedding (intfloat/multilingual-e5-large, alpha=0.5)
Accuracy : 0.9201
Hit@3    : 0.9896
Hit@5    : 0.9983
Hit@10   : 1.0000
MRR@10   : 0.9556


In [ ]:
# --- BGE-M3 embeddings on TF-IDF corpus ---
doc_embeddings_tfidf_bge, query_embeddings_tfidf_bge = build_or_load_embeddings(
    model_name=BGE_M3_MODEL_NAME,
    corpus_texts=corpus_texts_tfidf,
    query_texts=df_eval["query"].tolist(),
    cache_prefix="tfidf_corpus"
)

tfidf_bge_topk_preds = []

for i in range(len(df_eval)):
    emb_scores = np.dot(doc_embeddings_tfidf_bge, query_embeddings_tfidf_bge[i])
    tfidf_scores = tfidf_all_scores[i]

    tfidf_norm = minmax_scale(tfidf_scores) if np.max(tfidf_scores) != np.min(tfidf_scores) else np.zeros_like(tfidf_scores)
    emb_norm = minmax_scale(emb_scores) if np.max(emb_scores) != np.min(emb_scores) else np.zeros_like(emb_scores)

    final_scores = TFIDF_EMB_ALPHA * tfidf_norm + (1 - TFIDF_EMB_ALPHA) * emb_norm

    ranked_idx = np.argsort(final_scores)[::-1][:TOP_K]
    pred_ids = [corpus_ids_tfidf[idx] for idx in ranked_idx]
    tfidf_bge_topk_preds.append(pred_ids)

tfidf_bge_summary, tfidf_bge_best_preds, tfidf_bge_ranks = compute_metrics(
    y_true,
    tfidf_bge_topk_preds,
    f"TF-IDF + Embedding ({BGE_M3_MODEL_NAME}, alpha={TFIDF_EMB_ALPHA})"
)

tfidf_bge_details = build_detail_df(
    df_eval,
    tfidf_bge_topk_preds,
    tfidf_bge_best_preds,
    tfidf_bge_ranks,
    TOP_K
)

print_summary(tfidf_bge_summary)

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Encoding doc embeddings for BAAI/bge-m3 ...


Batches:   0%|          | 0/20 [00:00<?, ?it/s]

Encoding query embeddings for BAAI/bge-m3 ...


Batches:   0%|          | 0/18 [00:00<?, ?it/s]

TF-IDF + Embedding (BAAI/bge-m3, alpha=0.5)
Accuracy : 0.9149
Hit@3    : 0.9913
Hit@5    : 0.9983
Hit@10   : 1.0000
MRR@10   : 0.9536


#### BM25 + Embedding

In [ ]:
# ===== BM25 + EMBEDDING (E5 + BGE-M3) =====

# --- BM25 + E5 ---
bm25_e5_topk_preds = []

for bm25_scores, emb_scores in zip(bm25_all_scores, embedding_all_scores):
    bm25_norm = minmax_scale(bm25_scores) if np.max(bm25_scores) != np.min(bm25_scores) else np.zeros_like(bm25_scores)
    emb_norm = minmax_scale(emb_scores) if np.max(emb_scores) != np.min(emb_scores) else np.zeros_like(emb_scores)

    final_scores = BM25_EMB_ALPHA * bm25_norm + (1 - BM25_EMB_ALPHA) * emb_norm

    ranked_idx = np.argsort(final_scores)[::-1][:TOP_K]
    pred_ids = [corpus_ids_bm25[idx] for idx in ranked_idx]
    bm25_e5_topk_preds.append(pred_ids)

bm25_e5_summary, bm25_e5_best_preds, bm25_e5_ranks = compute_metrics(
    y_true,
    bm25_e5_topk_preds,
    f"BM25 + Embedding ({E5_MODEL_NAME}, alpha={BM25_EMB_ALPHA})"
)

bm25_e5_details = build_detail_df(
    df_eval,
    bm25_e5_topk_preds,
    bm25_e5_best_preds,
    bm25_e5_ranks,
    TOP_K
)

print_summary(bm25_e5_summary)

BM25 + Embedding (intfloat/multilingual-e5-large, alpha=0.5)
Accuracy : 0.9253
Hit@3    : 0.9965
Hit@5    : 0.9965
Hit@10   : 0.9983
MRR@10   : 0.9583


In [ ]:
# --- BM25 + BGE-M3 ---
doc_embeddings_bge_m3, query_embeddings_bge_m3 = build_or_load_embeddings(
    model_name=BGE_M3_MODEL_NAME,
    corpus_texts=corpus_texts_bm25,
    query_texts=df_eval["query"].tolist(),
    cache_prefix="bm25_corpus"
)

bge_m3_all_scores = []
for q_emb in query_embeddings_bge_m3:
    scores = np.dot(doc_embeddings_bge_m3, q_emb)
    bge_m3_all_scores.append(scores)

bm25_bge_topk_preds = []

for bm25_scores, emb_scores in zip(bm25_all_scores, bge_m3_all_scores):
    bm25_norm = minmax_scale(bm25_scores) if np.max(bm25_scores) != np.min(bm25_scores) else np.zeros_like(bm25_scores)
    emb_norm = minmax_scale(emb_scores) if np.max(emb_scores) != np.min(emb_scores) else np.zeros_like(emb_scores)

    final_scores = BM25_EMB_ALPHA * bm25_norm + (1 - BM25_EMB_ALPHA) * emb_norm

    ranked_idx = np.argsort(final_scores)[::-1][:TOP_K]
    pred_ids = [corpus_ids_bm25[idx] for idx in ranked_idx]
    bm25_bge_topk_preds.append(pred_ids)

bm25_bge_summary, bm25_bge_best_preds, bm25_bge_ranks = compute_metrics(
    y_true,
    bm25_bge_topk_preds,
    f"BM25 + Embedding ({BGE_M3_MODEL_NAME}, alpha={BM25_EMB_ALPHA})"
)

bm25_bge_details = build_detail_df(
    df_eval,
    bm25_bge_topk_preds,
    bm25_bge_best_preds,
    bm25_bge_ranks,
    TOP_K
)

print_summary(bm25_bge_summary)

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Loading cached doc embeddings for BAAI/bge-m3 ...
Loading cached query embeddings for BAAI/bge-m3 ...
BM25 + Embedding (BAAI/bge-m3, alpha=0.5)
Accuracy : 0.9149
Hit@3    : 0.9983
Hit@5    : 1.0000
Hit@10   : 1.0000
MRR@10   : 0.9552


#### EXPORT REPORT

In [ ]:
summary_df = pd.DataFrame([
    tfidf_summary,
    bm25_summary,
    embedding_summary,      # Embedding E5
    bge_m3_summary,         # Embedding BGE-M3
    tfidf_e5_summary,
    tfidf_bge_summary,
    bm25_e5_summary,
    bm25_bge_summary
])

with pd.ExcelWriter(OUTPUT_XLSX, engine="openpyxl") as writer:
    summary_df.to_excel(writer, sheet_name="summary", index=False)

    tfidf_details.to_excel(writer, sheet_name="tfidf_details", index=False)
    bm25_details.to_excel(writer, sheet_name="bm25_details", index=False)

    embedding_details.to_excel(writer, sheet_name="e5_details", index=False)
    bge_m3_details.to_excel(writer, sheet_name="bge_m3_details", index=False)

    tfidf_e5_details.to_excel(writer, sheet_name="tfidf_e5_details", index=False)
    tfidf_bge_details.to_excel(writer, sheet_name="tfidf_bge_details", index=False)

    bm25_e5_details.to_excel(writer, sheet_name="bm25_e5_details", index=False)
    bm25_bge_details.to_excel(writer, sheet_name="bm25_bge_details", index=False)

print("Saved to:", OUTPUT_XLSX)
display(summary_df)

Saved to: /content/drive/MyDrive/CS2231/kg/retrieval_method_comparison_full.xlsx


,model,Accuracy,Hit@3,Hit@5,Hit@10,MRR@10
0,TF-IDF,0.875000,0.979167,0.998264,1.000000,0.928588
1,BM25,0.895833,0.987847,0.998264,1.000000,0.942419
2,Embedding (intfloat/multilingual-e5-large),0.829861,0.984375,0.993056,0.996528,0.905527
3,Embedding (BAAI/bge-m3),0.821181,0.949653,0.972222,0.987847,0.887463
4,TF-IDF + Embedding (intfloat/multilingual-e5-l...,0.920139,0.989583,0.998264,1.000000,0.955584
5,"TF-IDF + Embedding (BAAI/bge-m3, alpha=0.5)",0.914931,0.991319,0.998264,1.000000,0.953617
6,BM25 + Embedding (intfloat/multilingual-e5-lar...,0.925347,0.996528,0.996528,0.998264,0.958333
7,"BM25 + Embedding (BAAI/bge-m3, alpha=0.5)",0.914931,0.998264,1.000000,1.000000,0.955208


## ==> Kết luận: Dựa vào bảng chọn BM25 + Embedding E5

## Evaluation LLMs

In [12]:
import json
import re
import time
import math
import gc
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
import torch

from sklearn.metrics import accuracy_score
from sklearn.preprocessing import minmax_scale

from rank_bm25 import BM25Okapi
from underthesea import word_tokenize
from sentence_transformers import SentenceTransformer
from tqdm.auto import tqdm

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
)

In [13]:
from huggingface_hub import login
login("HF_TOKEN_REMOVED")

In [ ]:
# ===== PATH =====
KG_DIR = Path("/content/drive/MyDrive/Colab Notebooks/kg")
KG_NODES_FILE = KG_DIR / "kg_nodes.jsonl"
TEST_CSV = KG_DIR / "test.csv"

EMB_CACHE_DIR = KG_DIR / "embedding_cache"
EMB_CACHE_DIR.mkdir(parents=True, exist_ok=True)

# ===== FIXED RETRIEVER =====
E5_MODEL_NAME = "intfloat/multilingual-e5-large"
TOP_K = 10
BM25_EMB_ALPHA = 0.5

# ===== CHỈ CHẠY 1 MODEL MỖI LẦN =====
# CURRENT_MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"
# CURRENT_MODEL_DISPLAY_NAME = "Qwen2.5-3B-Instruct"
# CURRENT_TRUST_REMOTE_CODE = False

# Ví dụ model khác:
# CURRENT_MODEL_NAME = "unsloth/Llama-3.2-3B"
# CURRENT_MODEL_DISPLAY_NAME = "Llama-3.2-3B"
# CURRENT_TRUST_REMOTE_CODE = False

# CURRENT_MODEL_NAME = "unsloth/gemma-2b-it"
# CURRENT_MODEL_DISPLAY_NAME = "Gemma-2B-It"
# CURRENT_TRUST_REMOTE_CODE = False

CURRENT_MODEL_NAME = "mistralai/Mistral-7B-Instruct-v0.3"
CURRENT_MODEL_DISPLAY_NAME = "Mistral-7B-Instruct-v0.3"
CURRENT_TRUST_REMOTE_CODE = False

# ===== GENERATION =====
USE_4BIT = True
DEVICE_MAP = "auto"
MAX_NEW_TOKENS = 32
DO_SAMPLE = False
TEMPERATURE = 0.0

In [15]:
def recall_at_k(ranks, k):
    return sum(1 for r in ranks if r is not None and r <= k) / len(ranks)

def precision_at_k(ranks, k):
    return sum((1 / k) if r is not None and r <= k else 0 for r in ranks) / len(ranks)

def mrr_at_k(ranks, k):
    return sum((1 / r) if r is not None and r <= k else 0 for r in ranks) / len(ranks)

def ndcg_at_k(ranks, k):
    return sum((1 / math.log2(r + 1)) if r is not None and r <= k else 0 for r in ranks) / len(ranks)

def compute_metrics(y_true, topk_preds, method_name, k=10):
    best_preds = [preds[0] if len(preds) > 0 else "NOT_FOUND" for preds in topk_preds]
    ranks = []

    for true_id, preds in zip(y_true, topk_preds):
        if true_id in preds:
            ranks.append(preds.index(true_id) + 1)
        else:
            ranks.append(None)

    accuracy = accuracy_score(y_true, best_preds)
    recall_k = recall_at_k(ranks, k)
    precision_k = precision_at_k(ranks, k)
    mrr_k = mrr_at_k(ranks, k)
    ndcg_k = ndcg_at_k(ranks, k)

    summary_row = {
        "model": method_name,
        "Accuracy": accuracy,
        f"Recall@{k}": recall_k,
        f"Precision@{k}": precision_k,
        f"MRR@{k}": mrr_k,
        f"nDCG@{k}": ndcg_k,
    }
    return summary_row, best_preds, ranks

def build_detail_df(df_eval, rewritten_queries, topk_preds, best_preds, ranks, latencies, top_k=10):
    details_df = df_eval.copy()
    details_df["violation_id"] = details_df["violation_id"].astype(str)
    details_df["rewritten_query"] = rewritten_queries
    details_df["top_1_pred"] = best_preds
    details_df["rank_of_true_in_top10"] = ranks
    details_df["latency_sec"] = latencies

    for i in range(top_k):
        details_df[f"top_{i+1}"] = [
            preds[i] if i < len(preds) else "" for preds in topk_preds
        ]

    details_df["is_correct_top1"] = details_df["violation_id"].astype(str) == details_df["top_1_pred"].astype(str)
    details_df["recall@10"] = details_df["rank_of_true_in_top10"].apply(lambda x: x is not None and x <= 10)
    details_df["precision@10"] = details_df["rank_of_true_in_top10"].apply(lambda x: (1 / 10) if x is not None and x <= 10 else 0)
    details_df["rr@10"] = details_df["rank_of_true_in_top10"].apply(lambda x: (1 / x) if x is not None and x <= 10 else 0)
    details_df["ndcg@10"] = details_df["rank_of_true_in_top10"].apply(
        lambda x: (1 / math.log2(x + 1)) if x is not None and x <= 10 else 0
    )
    return details_df

def print_summary(summary_row):
    print("=" * 50)
    print(summary_row["model"])
    print("=" * 50)
    print(f'Accuracy     : {summary_row["Accuracy"]:.4f}')
    print(f'Recall@10    : {summary_row["Recall@10"]:.4f}')
    print(f'Precision@10 : {summary_row["Precision@10"]:.4f}')
    print(f'MRR@10       : {summary_row["MRR@10"]:.4f}')
    print(f'nDCG@10      : {summary_row["nDCG@10"]:.4f}')

In [16]:
def norm_bm25(text):
    if text is None:
        return ""
    text = str(text).lower().strip()
    text = re.sub(r"[^\w\s]", " ", text, flags=re.UNICODE)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def tokenize_bm25(text: str):
    text = norm_bm25(text)
    segmented = word_tokenize(text, format="text")
    return segmented.split()

def safe_str(x):
    if x is None:
        return ""
    return str(x).strip()

def make_violation_text_bm25(node):
    parts = [
        safe_str(node.get("description_natural")),
        safe_str(node.get("normalized_violation")),
        safe_str(node.get("vehicle_type")),
        safe_str(node.get("context_condition")),
    ]
    return " ".join([p for p in parts if p])

In [17]:
# ===== BUILD BM25 / E5 CORPUS =====
violation_rows = []

with open(KG_NODES_FILE, "r", encoding="utf-8") as f:
    for line in f:
        node = json.loads(line)
        if node.get("type") != "Violation":
            continue

        violation_id = safe_str(node.get("violation_id"))
        if not violation_id:
            continue

        text = make_violation_text_bm25(node)

        violation_rows.append({
            "node_id": node.get("id"),
            "violation_id": violation_id,
            "text": text,
            "raw_node": node,
        })

df_corpus = (
    pd.DataFrame(violation_rows)
    .drop_duplicates(subset=["violation_id"])
    .reset_index(drop=True)
)

print("Corpus size:", len(df_corpus))
display(df_corpus.head())

corpus_ids = df_corpus["violation_id"].tolist()
corpus_texts = df_corpus["text"].tolist()

# ===== LOAD TEST =====
df_eval = pd.read_csv(TEST_CSV)
df_eval["violation_id"] = df_eval["violation_id"].astype(str)
df_eval["query"] = df_eval["query"].astype(str)

print("Test size:", len(df_eval))
display(df_eval.head())

y_true = df_eval["violation_id"].tolist()

Corpus size: 613


,node_id,violation_id,text,raw_node
0,VIOLATION_V001,V001,không có đèn chiếu sáng 40 hoặc tấm phản quang...,"{'id': 'VIOLATION_V001', 'type': 'Violation', ..."
1,VIOLATION_V002,V002,"Không chấp hành hướng dẫn của lái xe, nhân viê...","{'id': 'VIOLATION_V002', 'type': 'Violation', ..."
2,VIOLATION_V003,V003,"Không đi bên phải theo chiều đi của mình, đi k...","{'id': 'VIOLATION_V003', 'type': 'Violation', ..."
3,VIOLATION_V004,V004,Dừng xe đột ngột; chuyển hướng không báo hiệu ...,"{'id': 'VIOLATION_V004', 'type': 'Violation', ..."
4,VIOLATION_V005,V005,Không chấp hành hiệu lệnh hoặc chỉ dẫn của biể...,"{'id': 'VIOLATION_V005', 'type': 'Violation', ..."


Test size: 576


,query,behavior,legal,fine_min,fine_max,violation_id
0,Tôi muốn tra lỗi xe thô sơ: không có đèn chiếu...,không có đèn chiếu sáng 40 hoặc tấm phản quang...,corpus://dataset-traffic-law#dieu-15-khoan-1-d...,100000,200000,V001
1,Cho tôi hỏi hành vi Không chấp hành hướng dẫn ...,"Không chấp hành hướng dẫn của lái xe, nhân viê...",corpus://dataset-traffic-law#dieu-33-khoan-1-d...,100000,200000,V002
2,Tra cứu vi phạm giao thông: xe đạp Không đi bê...,"Không đi bên phải theo chiều đi của mình, đi k...",corpus://dataset-traffic-law#dieu-9-khoan-1-di...,100000,200000,V003
3,Trường hợp xe đạp Dừng xe đột ngột thì bị xử p...,Dừng xe đột ngột; chuyển hướng không báo hiệu ...,corpus://dataset-traffic-law#dieu-9-khoan-1-di...,100000,200000,V004
4,Hỏi nhanh lỗi Không chấp hành hiệu lệnh hoặc c...,Không chấp hành hiệu lệnh hoặc chỉ dẫn của biể...,corpus://dataset-traffic-law#dieu-9-khoan-1-di...,100000,200000,V005


#### Build E5 + BM25

In [18]:
# ===== BM25 =====
tokenized_corpus = [tokenize_bm25(text) for text in corpus_texts]
bm25 = BM25Okapi(tokenized_corpus)

# ===== E5 EMBEDDINGS =====
e5_model = SentenceTransformer(E5_MODEL_NAME)

doc_emb_path = EMB_CACHE_DIR / "e5_large_doc_embeddings_for_llm_eval.npy"
passages = [f"passage: {text}" for text in corpus_texts]

if doc_emb_path.exists():
    print("Loading cached E5 doc embeddings...")
    doc_embeddings = np.load(doc_emb_path)
else:
    print("Encoding E5 doc embeddings...")
    doc_embeddings = e5_model.encode(
        passages,
        normalize_embeddings=True,
        convert_to_numpy=True,
        batch_size=32,
        show_progress_bar=True
    ).astype("float32")
    np.save(doc_emb_path, doc_embeddings)

print("Retriever ready.")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/690 [00:00<?, ?B/s]

...

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/201 [00:00<?, ?B/s]

Loading cached E5 doc embeddings...
Retriever ready.


In [19]:
def hybrid_retrieve(query: str, top_k: int = 10, alpha: float = 0.5):
    # BM25
    tokenized_query = tokenize_bm25(query)
    bm25_scores = bm25.get_scores(tokenized_query)

    # E5
    q_emb = e5_model.encode(
        [f"query: {query}"],
        normalize_embeddings=True,
        convert_to_numpy=True
    ).astype("float32")[0]
    emb_scores = np.dot(doc_embeddings, q_emb)

    # Normalize + combine
    bm25_norm = minmax_scale(bm25_scores) if np.max(bm25_scores) != np.min(bm25_scores) else np.zeros_like(bm25_scores)
    emb_norm = minmax_scale(emb_scores) if np.max(emb_scores) != np.min(emb_scores) else np.zeros_like(emb_scores)

    final_scores = alpha * bm25_norm + (1 - alpha) * emb_norm

    ranked_idx = np.argsort(final_scores)[::-1][:top_k]
    pred_ids = [corpus_ids[idx] for idx in ranked_idx]

    return pred_ids

#### Prompt

In [20]:
SYSTEM_PROMPT = """Bạn là bộ tiền xử lý truy vấn cho hệ thống tra cứu luật giao thông Việt Nam.

Nhiệm vụ:
- Đọc câu hỏi tự nhiên của người dùng.
- Viết lại thành 1 truy vấn ngắn, rõ, sát hành vi vi phạm để truy xuất knowledge graph.
- Giữ lại phương tiện, hành vi vi phạm, điều kiện ngữ cảnh nếu có.
- Không trả lời tư vấn pháp lý.
- Không giải thích dài dòng.

Bắt buộc chỉ trả về JSON hợp lệ theo đúng format:
{"rewritten_query":"..."}
"""

def build_messages(user_query: str):
    user_prompt = f"""Câu hỏi người dùng:
{user_query}

Hãy rút gọn thành truy vấn tìm kiếm pháp lý ngắn gọn bằng tiếng Việt.
Chỉ trả về JSON."""
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_prompt},
    ]

def extract_json_text(text: str):
    text = text.strip()
    match = re.search(r"\{.*\}", text, flags=re.DOTALL)
    if match:
        return match.group(0)
    return text

def parse_rewritten_query(raw_output: str, fallback_query: str):
    try:
        raw_json = extract_json_text(raw_output)
        data = json.loads(raw_json)
        rewritten = str(data.get("rewritten_query", "")).strip()
        if rewritten:
            return rewritten
    except Exception:
        pass

    cleaned = raw_output.strip().split("\n")[0].strip()
    if cleaned.startswith("{") and cleaned.endswith("}"):
        try:
            data = json.loads(cleaned)
            rewritten = str(data.get("rewritten_query", "")).strip()
            if rewritten:
                return rewritten
        except Exception:
            pass

    return fallback_query

In [21]:
def free_cuda():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        try:
            torch.cuda.ipc_collect()
        except Exception:
            pass

def load_llm(model_name: str, trust_remote_code: bool = False):
    free_cuda()

    quant_config = None
    if USE_4BIT:
        quant_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_use_double_quant=True,
        )

    tokenizer = AutoTokenizer.from_pretrained(
        CURRENT_MODEL_NAME,
        token='HF_TOKEN_REMOVED',
        trust_remote_code=CURRENT_TRUST_REMOTE_CODE,
    )


    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    model = AutoModelForCausalLM.from_pretrained(
        CURRENT_MODEL_NAME,
        token='HF_TOKEN_REMOVED',
        dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
        device_map=DEVICE_MAP,
        quantization_config=quant_config,
        trust_remote_code=CURRENT_TRUST_REMOTE_CODE,
    )
    model.eval()
    return tokenizer, model

def unload_llm(tokenizer=None, model=None):
    try:
        del tokenizer
    except Exception:
        pass
    try:
        del model
    except Exception:
        pass
    free_cuda()

In [22]:
def generate_rewrite(tokenizer, model, user_query: str):
    messages = build_messages(user_query)

    try:
        prompt = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )
    except Exception:
        # fallback plain prompt
        prompt = SYSTEM_PROMPT + "\n\nUser query: " + user_query + '\n\n{"rewritten_query":"'

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=2048
    )

    if torch.cuda.is_available():
        inputs = {k: v.to(model.device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=DO_SAMPLE,
            temperature=TEMPERATURE,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    gen_tokens = outputs[0][inputs["input_ids"].shape[1]:]
    raw_text = tokenizer.decode(gen_tokens, skip_special_tokens=True)

    rewritten = parse_rewritten_query(raw_text, fallback_query=user_query)
    return rewritten, raw_text

In [23]:
def evaluate_current_model():
    tokenizer, model = load_llm(
        CURRENT_MODEL_NAME,
        trust_remote_code=CURRENT_TRUST_REMOTE_CODE
    )

    rewritten_queries = []
    raw_outputs = []
    topk_preds = []
    latencies = []

    print(f"Evaluating {CURRENT_MODEL_DISPLAY_NAME} ...")

    for query in tqdm(df_eval["query"].tolist(), desc=CURRENT_MODEL_DISPLAY_NAME):
        start = time.time()

        rewritten_query, raw_output = generate_rewrite(tokenizer, model, query)
        pred_ids = hybrid_retrieve(rewritten_query, top_k=TOP_K, alpha=BM25_EMB_ALPHA)

        elapsed = time.time() - start

        rewritten_queries.append(rewritten_query)
        raw_outputs.append(raw_output)
        topk_preds.append(pred_ids)
        latencies.append(elapsed)

    summary, best_preds, ranks = compute_metrics(
        y_true, topk_preds, CURRENT_MODEL_DISPLAY_NAME, k=10
    )
    summary["avg_latency_sec"] = float(np.mean(latencies))

    details = build_detail_df(
        df_eval=df_eval,
        rewritten_queries=rewritten_queries,
        topk_preds=topk_preds,
        best_preds=best_preds,
        ranks=ranks,
        latencies=latencies,
        top_k=TOP_K
    )
    details["raw_llm_output"] = raw_outputs

    print_summary(summary)
    print(f"avg_latency_sec : {summary['avg_latency_sec']:.4f}")

    unload_llm(tokenizer, model)
    return summary, details

#### Baseline no LLM

In [ ]:
baseline_topk_preds = []
baseline_rewrites = []
baseline_latencies = []

for query in tqdm(df_eval["query"].tolist(), desc="Baseline raw query"):
    start = time.time()
    pred_ids = hybrid_retrieve(query, top_k=TOP_K, alpha=BM25_EMB_ALPHA)
    elapsed = time.time() - start

    baseline_topk_preds.append(pred_ids)
    baseline_rewrites.append(query)
    baseline_latencies.append(elapsed)

baseline_summary, baseline_best_preds, baseline_ranks = compute_metrics(
    y_true, baseline_topk_preds, "No LLM (raw query)", k=10
)
baseline_summary["avg_latency_sec"] = float(np.mean(baseline_latencies))

baseline_details = build_detail_df(
    df_eval=df_eval,
    rewritten_queries=baseline_rewrites,
    topk_preds=baseline_topk_preds,
    best_preds=baseline_best_preds,
    ranks=baseline_ranks,
    latencies=baseline_latencies,
    top_k=TOP_K
)
baseline_details["raw_llm_output"] = ""

print_summary(baseline_summary)
print(f"avg_latency_sec : {baseline_summary['avg_latency_sec']:.4f}")

Baseline raw query:   0%|          | 0/576 [00:00<?, ?it/s]

No LLM (raw query)
Accuracy     : 0.9253
Recall@10    : 0.9983
Precision@10 : 0.0998
MRR@10       : 0.9583
nDCG@10      : 0.9686
avg_latency_sec : 0.0850


#### Qwen2.5 3B

In [ ]:
current_summary, current_details = evaluate_current_model()

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

Evaluating Qwen2.5-3B-Instruct ...


Qwen2.5-3B-Instruct:   0%|          | 0/576 [00:00<?, ?it/s]

Qwen2.5-3B-Instruct
Accuracy     : 0.8889
Recall@10    : 0.9983
Precision@10 : 0.0998
MRR@10       : 0.9356
nDCG@10      : 0.9515
avg_latency_sec : 3.3353


#### Llama-3.2-3B

In [ ]:
current_summary, current_details = evaluate_current_model()

config.json:   0%|          | 0.00/890 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/459 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

...

generation_config.json:   0%|          | 0.00/230 [00:00<?, ?B/s]

Evaluating Llama-3.2-3B ...


Llama-3.2-3B:   0%|          | 0/576 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Llama-3.2-3B
Accuracy     : 0.9253
Recall@10    : 0.9983
Precision@10 : 0.0998
MRR@10       : 0.9583
nDCG@10      : 0.9686
avg_latency_sec : 2.6480


#### Gemma-2B-It

In [ ]:
current_summary, current_details = evaluate_current_model()

config.json:   0%|          | 0.00/724 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.5M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/5.01G [00:00<?, ?B/s]

...

generation_config.json:   0%|          | 0.00/154 [00:00<?, ?B/s]

Evaluating Gemma-2B-It ...


Gemma-2B-It:   0%|          | 0/576 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Gemma-2B-It
Accuracy     : 0.9253
Recall@10    : 0.9983
Precision@10 : 0.0998
MRR@10       : 0.9583
nDCG@10      : 0.9686
avg_latency_sec : 1.5413


#### Ministral-3-3B-Instruct

In [24]:
current_summary, current_details = evaluate_current_model()

config.json:   0%|          | 0.00/601 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/587k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

...

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

Evaluating Mistral-7B-Instruct-v0.3 ...


Mistral-7B-Instruct-v0.3:   0%|          | 0/576 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Mistral-7B-Instruct-v0.3
Accuracy     : 0.9236
Recall@10    : 0.9983
Precision@10 : 0.0998
MRR@10       : 0.9575
nDCG@10      : 0.9680
avg_latency_sec : 3.3274
